# Rapport MS COCO classification challenge

To run this notebook successfully, you need to have the MS COCO dataset downloaded and every python package installed. All python requirements are listed in the appropriate files given in the zip file and was created using uv. You can also create an .env file to provide different variables to the notebook using .env.example as a template.

**Plan of the report**

| Section | Content | Run it? |
|---|---|---|
| 1-3 | Imports, configuration, dataset files | always |
| 4 | Analysis of the annotations and of the weight of each class in the leaderboard metric | always |
| 5-6 | Dataset, split, loaders, near-duplicate diagnostic, **split integrity check** | always |
| 7 | Metrics (leaderboard F1, thresholds, mAP) | always |
| 8-9 | Models, losses, training loop, **table of the methods and what they bring** | always |
| 10 | **Experiments** (long: about one hour per fine-tuned ResNet18 on the `mps` device) | only the runs listed in `RUN` that are not finished yet |
| 11-12 | Results table and curves, ensemble and thresholds | after training |
| 13-15 | Per-class metrics, test submission, submission check | after 12 |
| 16 | **Comparative analysis** | after training; needs no GPU |

**Which number to read.** `lb_f1_global` (leaderboard F1 with the best single threshold) is the headline number to compare models. `lb_f1_05` shows how good the raw probabilities are without any tuning. `lb_f1_heldout` is the honest estimate of the final pipeline. Details in section 7.

## 1. Installation and imports

We start by importing all our libraries needed for this notebook. We also check the version of Python and PyTorch, and we initialize the device to be used (CPU or GPU). Setting DEVICE in the .env is not compulsory as this code will try to detect the best device available. If you want to force the use of a specific device, you can set the DEVICE variable in the .env file to `cpu`, `cuda`, or `mps` for example. If there is an issue with the execution directory, you can also set the `DIR_PATH` variable in your environment variable or change it's value directly in the code below.

In [ ]:
import os
import json
import time
import copy
import gc
import platform
import random
import contextlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision

from dotenv import load_dotenv
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

pwd = os.getenv("DIR_PATH", "/Users/ProjetPIR/PR_CR/theo/model")
os.chdir(pwd) if pwd else None

import sys

PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

from lib.util import CLASSES, COCOTrainImageDataset, COCOTestImageDataset, TransformSubset

load_dotenv()

# device initialization
device_name = os.getenv("DEVICE", "")
DEVICE = torch.device(
    device_name) if device_name else (
    torch.accelerator.current_accelerator()) if (
    torch.accelerator.is_available()) else (
    torch.device("cpu"))

print("Python :", platform.python_version())
print("PyTorch :", torch.__version__)
print("Device used :", DEVICE)
print("Execution directory :", os.getcwd())

## 2. Configuration

Same idea as before (paths, seed, batch size...), plus the options of the pipeline: mixed precision (CUDA only), model EMA, flip test-time augmentation (TTA), repeat-factor sampling threshold and the group-aware split. Image size is no longer global: it is set per experiment in section 10.

New in v4: `SELECTION_METRIC` is the metric used to keep the best epoch of every run. The default is `lb_f1_global`, i.e. the leaderboard metric (inverse-frequency-weighted F1) computed with the best single threshold on validation. The alternatives are `lb_f1_05` (fixed threshold 0.5) and `mAP_w` (threshold-free).

In [ ]:
# Defining global variables for data directories
DATA_ROOT = Path(os.getenv("DATA_ROOT", "/opt/iav/labs/data"))
DATA_ROOT = DATA_ROOT / "ms-coco"
IMAGE_DIR = DATA_ROOT / "images/train"
LABEL_DIR = DATA_ROOT / "labels/train"
TEST_IMAGE_DIR = DATA_ROOT / "images/test"

# Defining global variables for output directories
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_PATH = OUTPUT_DIR / os.getenv("JSON_PATH", "submission.json")

SEED = 30
EPOCHS = int(os.getenv("EPOCHS", 15))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", 32))
NUM_WORKERS = min(8, os.cpu_count() or 1)

# Fraction of the dataset to be used for validation
VAL_FRACTION = 0.15
# Split train/val by groups of near-duplicate images (see section 6)
DEDUP_SPLIT = True

# Default threshold used only for the simple "@0.5" reference metrics
THRESHOLD = 0.5

# For a quick run, you can set QUICK_RUN to True and limit the number of images processed
# (delete the cached files in OUTPUT_DIR when you switch this flag)
QUICK_RUN = False
QUICK_MAX_IMAGES = 5000

# Speed / training options
PIN_MEMORY = DEVICE.type == "cuda"
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
CHANNELS_LAST = DEVICE.type == "cuda"
USE_EMA = True
EMA_DECAY = 0.999
GRAD_CLIP = 5.0
USE_TTA_FLIP = True  # average the prediction of the image and of its horizontal flip
REPEAT_T = 0.02  # repeat-factor sampling: classes in fewer than 2% of the images are oversampled

# Metric used to keep the best epoch of every run:
#   "lb_f1_global" = leaderboard F1 with the best single threshold, "lb_f1_05" = leaderboard F1 at threshold 0.5,
#   "mAP_w" = weighted mAP (threshold-free)
SELECTION_METRIC = "lb_f1_global"
assert SELECTION_METRIC in ("lb_f1_global", "lb_f1_05", "mAP_w")


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE) if USE_AMP else contextlib.nullcontext()


# Function to set random seed for reproducibility
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = (DEVICE.type == "cuda")


set_seed()

print("Images :", IMAGE_DIR.resolve())
print("Annotations :", LABEL_DIR.resolve())
print("Test images :", TEST_IMAGE_DIR.resolve())
print("Outputs :", OUTPUT_DIR.resolve())
print("Submission :", JSON_PATH.resolve())
print("AMP :", USE_AMP, AMP_DTYPE if USE_AMP else "")
print("Selection metric :", SELECTION_METRIC)

## 3. Classes definitions and files verification

Here we define the classes and verify that we can access the images and annotations. If everything works fine, you should see the number of images and annotations found, and they should be equal to each other (Normally, there is 65 000 images if the dataset is correctly downloaded). You should also see an example of an image and its corresponding annotation file.

In [ ]:
classes = CLASSES
NUM_CLASSES = len(classes)
assert NUM_CLASSES == 80, f"We expect to find 80 classes but {NUM_CLASSES} were found."

if not IMAGE_DIR.is_dir() or not LABEL_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset not found. Verify DATA_ROOT={DATA_ROOT.resolve()} ; "
        f"There must be a folder 'images/train' and a folder 'labels/train' containing the .jpg images and .cls annotation files respectively."
    )

test_images = sorted(IMAGE_DIR.glob("*.jpg"))
label_files = sorted(LABEL_DIR.glob("*.cls"))

if not test_images:
    raise RuntimeError("No .jpg files found in IMAGE_DIR.")
if not label_files:
    raise RuntimeError("No .cls files found in LABEL_DIR.")

print(f"Images : {len(test_images):,}")
print(f"Annotated files : {len(label_files):,}")
print("Image example :", test_images[0].name if test_images else None)
print("annotation example :", label_files[0].name if label_files else None)

if len(test_images) != len(label_files):
    print(
        f"Warning: Number of images ({len(test_images)}) does not match number of labels ({len(label_files)}). Some images may not have corresponding labels or vice versa.")

## 4. Annotations analysis

Before going straight to the training part, we will analyze the annotations to see the distribution of classes before any training in order to analyse the composition of the dataset.

In [ ]:
scan_files = label_files[:QUICK_MAX_IMAGES] if QUICK_RUN else label_files
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
labels_per_image = []
invalid_rows = []

for p in tqdm(scan_files, desc="Annotations analysis"):
    try:
        ids = [int(x.strip()) for x in p.read_text().splitlines() if x.strip()]
        if any(i < 0 or i >= NUM_CLASSES for i in ids):
            invalid_rows.append(p.name)
            continue
        class_counts[np.unique(ids)] += 1
        labels_per_image.append(len(set(ids)))
    except Exception as e:
        print(f"Error processing {p.name}: {e}")
        invalid_rows.append(p.name)

print("Images analysed :", len(scan_files))
print("Annotations problematic :", len(invalid_rows))
print("Average number of classes/image :", round(float(np.mean(labels_per_image)), 3))
print("Median :", float(np.median(labels_per_image)))
print("Min : ", min(labels_per_image))
print("Max : ", max(labels_per_image))

# Displaying the distribution of images per class in a DataFrame
freq_df = pd.DataFrame({"class": classes, "images_positives": class_counts})
display(freq_df.sort_values("images_positives"))

# Distribution of the number of images per class
plt.figure(figsize=(16, 4))
plt.bar(np.arange(NUM_CLASSES), class_counts)
plt.title("Number of positive images per class")
plt.xlabel("Class identifier")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()

# Distribution of the number of classes per image
plt.figure(figsize=(7, 4))
plt.hist(labels_per_image, bins=range(1, max(labels_per_image) + 2), align="left")
plt.title("Number of classes per image")
plt.xlabel("Number of classes")
plt.ylabel("Images")
plt.tight_layout()
plt.show()

As we can see, the distribution of classes is highly imbalanced. Firstly, the class "person" is the most represented class by far with 37,000 images. This means that the dataset is heavily biased towards images containing people, which could potentially lead to biased model predictions if not addressed properly. We also have classes with very few images, such as `hair drier` or `toaster` with only roughly 100 images. This extreme imbalance can make it difficult for the model to learn to recognize these underrepresented classes, leading to poor performance on them. The average number of classes per image is around three, which indicates that most images contain multiple objects without overflowing with too many classes. The median number of classes per image is two, which means that half of the images contain two or fewer classes going in the direction of a low number of classes per image.

### Which classes really drive the leaderboard score?

The leaderboard metric (`inverse_frequency_weighted_f1`) averages precision and recall with weights proportional to `1 / n_positives`, then takes the F1 of these two averages. So a class with 100 images weighs about 350 times more than `person`. The next cell shows how concentrated the metric is: the model must be good on the rare classes, and a very high global accuracy on frequent classes barely moves the score.

In [ ]:
metric_w = 1.0 / np.maximum(class_counts, 1)
metric_w = metric_w / metric_w.sum()
share_df = pd.DataFrame({"class": classes, "images_positives": class_counts, "metric_weight": metric_w})
share_df = share_df.sort_values("metric_weight", ascending=False).reset_index(drop=True)
share_df["cumulative_weight"] = share_df["metric_weight"].cumsum()
display(share_df.head(15))
print(f"The 10 rarest classes carry {share_df['metric_weight'].head(10).sum():.0%} of the leaderboard metric.")

## 5. Dataset PyTorch

The PyTorch Dataset classes for training and testing are defined in `lib/util.py` and imported at the beginning of this notebook. These classes will handle loading the images and their corresponding labels, applying any necessary transformations, and preparing the data for training and evaluation. They are based on the provided function given in the starting notebook.

## 6. Split, transformations and DataLoaders

**Why the split changed.** In the first notebook, validation F1 macro reached 0.885 (leaderboard-style F1 of 0.665) while the leaderboard gives about 0.44. A gap this large usually means that the validation set is *easier* than the test set, for example because near-identical images (same photo re-exported, flipped, slightly cropped...) end up on both sides of a random split. In that case every choice made on validation (best epoch, thresholds, model selection) is biased.

The cells below (1) compute a cheap 64-bit perceptual hash of every image (also of its horizontal flip), (2) measure how many validation images have a near-duplicate in the training split, and how many test images have one in the train set, and (3) build the split **by groups of near-duplicates**. If the dataset has no duplicates, the groups are singletons and this is simply a random split.

**Other changes.**
- Train transform: same light augmentation as before (flip + light colour jitter). No `RandomResizedCrop`, because a crop can remove an object that is still in the label (label noise).
- Validation / test: the whole image is resized (no centre crop), so objects on the borders are never cut out.
- Image size is chosen per experiment; the loaders are rebuilt for each one but always use the same split.
- Optional repeat-factor sampling (LVIS-style): an image is repeated `r = max over its classes of max(1, sqrt(t / f_c))` times on average, `f_c` being the fraction of images containing class `c`. The validation set is never resampled, so thresholds tuned on it stay valid for the real distribution.

**What the diagnostic showed on the full dataset.** Only 0.9% of the validation images of a plain random split have a near-duplicate in the training split, and only 0.6% of the test images have one in the train folder. Near-duplicates are therefore *not* what separates a validation F1 close to 0.99 (section 11) from a leaderboard score of about 0.44 (first notebook). The group-aware split is kept because it is free (it removes that 0.9%), but it should not be presented as the explanation of the gap. The last cell of this section checks the other usual suspect: that the loaders really serve disjoint images.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
max_images = QUICK_MAX_IMAGES if QUICK_RUN else None


def make_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return train_tf, eval_tf


# ---------- near-duplicate detection (64-bit dHash, flip aware) ----------
def compute_dhash(dataset, batch_size=128):
    """Return (bits, bits_flip), two boolean arrays (N, 64): dHash of each image and of its horizontal flip."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    bits, bits_flip = [], []
    for batch in tqdm(loader, desc="dHash"):
        g = batch[0][:, 0]  # (B, 8, 9) grayscale
        bits.append((g[:, :, 1:] > g[:, :, :-1]).reshape(len(g), 64).numpy())
        gf = torch.flip(g, dims=[2])
        bits_flip.append((gf[:, :, 1:] > gf[:, :, :-1]).reshape(len(g), 64).numpy())
    return np.concatenate(bits), np.concatenate(bits_flip)


def near_duplicate_pairs(bits_a, bits_b, same, max_dist=3, max_bucket=200):
    """Pairs (i, j) with Hamming distance <= max_dist. Pigeonhole: with 4 chunks of 16 bits, at most 3 differing
    bits means at least one identical chunk, so we only compare images that share a chunk."""
    w = (1 << np.arange(16)).astype(np.int64)
    key = lambda b: b.reshape(len(b), 4, 16).astype(np.int64) @ w
    ka, kb = key(bits_a), key(bits_b)
    cand = set()
    for ch in range(4):
        buckets = {}
        for j, k in enumerate(kb[:, ch]):
            buckets.setdefault(int(k), []).append(j)
        for i, k in enumerate(ka[:, ch]):
            bucket = buckets.get(int(k), ())
            if len(bucket) > max_bucket:
                continue  # flat / blank images: skip degenerate buckets
            for j in bucket:
                if not same or j > i:
                    cand.add((i, j))
    if not cand:
        return np.zeros((0, 2), dtype=np.int64)
    cand = np.array(sorted(cand), dtype=np.int64)
    dist = (bits_a[cand[:, 0]] != bits_b[cand[:, 1]]).sum(1)
    return cand[dist <= max_dist]


def duplicate_pairs_flip_aware(bits, bits_flip):
    n = len(bits)
    both = np.concatenate([bits, bits_flip])  # image and flipped image are the same photo
    p = near_duplicate_pairs(both, both, same=True)
    p = np.stack([p[:, 0] % n, p[:, 1] % n], axis=1) if len(p) else p
    p = p[p[:, 0] != p[:, 1]] if len(p) else p
    return np.unique(np.sort(p, axis=1), axis=0) if len(p) else p


def group_ids_from_pairs(n, pairs):
    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[rj] = ri
    return np.array([find(i) for i in range(n)])


def grouped_split(groups, val_fraction, seed):
    rng = np.random.RandomState(seed)
    uniq, counts = np.unique(groups, return_counts=True)
    order = rng.permutation(len(uniq))
    target, val_groups, total = int(len(groups) * val_fraction), [], 0
    for k in order:
        if total >= target:
            break
        val_groups.append(uniq[k])
        total += counts[k]
    is_val = np.isin(groups, val_groups)
    return np.flatnonzero(~is_val), np.flatnonzero(is_val)


def leak_rate(pairs, train_idx, val_idx, n):
    side = np.zeros(n, dtype=np.int8)
    side[train_idx], side[val_idx] = 1, 2
    leaked = np.zeros(n, dtype=bool)
    for i, j in pairs:
        if side[i] == 2 and side[j] == 1: leaked[i] = True
        if side[j] == 2 and side[i] == 1: leaked[j] = True
    return leaked[val_idx].mean() if len(val_idx) else 0.0


# ---------- dataset + hashes (cached) ----------
base_dataset = COCOTrainImageDataset(IMAGE_DIR, LABEL_DIR, transform=None, maximum_images=max_images)
N_TOTAL = len(base_dataset)
hash_tf = transforms.Compose([transforms.Resize((8, 9)), transforms.Grayscale(), transforms.ToTensor()])

train_hash_path = OUTPUT_DIR / "dhash_train.npz"
if train_hash_path.exists() and len(np.load(train_hash_path)["bits"]) == N_TOTAL:
    h = np.load(train_hash_path);
    tr_bits, tr_bits_flip = h["bits"], h["bits_flip"]
else:
    tr_bits, tr_bits_flip = compute_dhash(
        COCOTrainImageDataset(IMAGE_DIR, LABEL_DIR, transform=hash_tf, maximum_images=max_images))
    np.savez_compressed(train_hash_path, bits=tr_bits, bits_flip=tr_bits_flip)

dup_pairs = duplicate_pairs_flip_aware(tr_bits, tr_bits_flip)
groups = group_ids_from_pairs(N_TOTAL, dup_pairs)
print(f"Near-duplicate pairs inside the training folder : {len(dup_pairs):,} "
      f"| images belonging to a group of size > 1 : {(np.bincount(groups)[groups] > 1).mean():.1%}")

# ---------- how bad would a plain random split be ? ----------
n_val = max(1, int(N_TOTAL * VAL_FRACTION))
generator = torch.Generator().manual_seed(SEED)
rand_train, rand_val = torch.utils.data.random_split(base_dataset, [N_TOTAL - n_val, n_val], generator=generator)
rand_train_idx, rand_val_idx = np.array(rand_train.indices), np.array(rand_val.indices)
print(f"Random split : {leak_rate(dup_pairs, rand_train_idx, rand_val_idx, N_TOTAL):.1%} of the validation images "
      f"have a near-duplicate in the training split")

# ---------- how similar is the TEST set to the train set ? ----------
if TEST_IMAGE_DIR.is_dir():
    test_hash_path = OUTPUT_DIR / "dhash_test.npz"
    if test_hash_path.exists():
        h = np.load(test_hash_path);
        te_bits, te_bits_flip = h["bits"], h["bits_flip"]
    else:
        te_bits, te_bits_flip = compute_dhash(COCOTestImageDataset(TEST_IMAGE_DIR, transform=hash_tf))
        np.savez_compressed(test_hash_path, bits=te_bits, bits_flip=te_bits_flip)
    tp = near_duplicate_pairs(np.concatenate([te_bits, te_bits_flip]), tr_bits, same=False)
    test_with_dup = len(np.unique(tp[:, 0] % len(te_bits))) / len(te_bits) if len(tp) else 0.0
    print(f"Test images with a near-duplicate in the train folder : {test_with_dup:.1%}")
    print("-> if the random-split leak is much larger than this number, validation scores are inflated.")

# ---------- final split ----------
if DEDUP_SPLIT:
    train_idx, val_idx = grouped_split(groups, VAL_FRACTION, SEED)
else:
    train_idx, val_idx = rand_train_idx, rand_val_idx
train_subset = torch.utils.data.Subset(base_dataset, train_idx.tolist())
val_subset = torch.utils.data.Subset(base_dataset, val_idx.tolist())
print(
    f"Split ({'group-aware' if DEDUP_SPLIT else 'random'}) -> train : {len(train_subset):,} | validation : {len(val_subset):,}")

# ---------- training targets (for sampler, loss, bias init) ----------
_, _eval_tf_small = make_transforms(32)
_tmp = TransformSubset(train_subset, _eval_tf_small)
if hasattr(_tmp, "targets"):
    TRAIN_TARGETS = np.asarray(_tmp.targets, dtype=np.float32)
else:
    TRAIN_TARGETS = np.concatenate([y.numpy() for _, y in
                                    DataLoader(_tmp, batch_size=256, shuffle=False, num_workers=NUM_WORKERS)])
assert TRAIN_TARGETS.shape == (len(train_subset), NUM_CLASSES), TRAIN_TARGETS.shape
del _tmp

_prior = np.clip(TRAIN_TARGETS.mean(0), 1e-4, 0.5)
PRIOR_LOGITS = np.log(_prior / (1 - _prior)).astype(np.float32)  # head bias init = log-odds of each class


def repeat_factor_weights(targets, t=REPEAT_T):
    freq = targets.mean(0)
    r_class = np.maximum(1.0, np.sqrt(t / np.maximum(freq, 1e-12)))
    r_img = (targets * r_class[None, :]).max(1)
    return np.where(r_img == 0, 1.0, r_img)


_w = repeat_factor_weights(TRAIN_TARGETS)
_rare = np.argsort(TRAIN_TARGETS.sum(0))[:5]
_eff = (_w[:, None] * TRAIN_TARGETS).sum(0) / _w.sum() * len(_w)
print("Rarest classes, images seen per epoch (raw -> with repeat-factor sampling):")
for c in _rare:
    print(f"  {classes[c]:<14s} {int(TRAIN_TARGETS[:, c].sum()):>5d} -> {_eff[c]:.0f}")


def make_loaders(img_size, use_rfs):
    train_tf, eval_tf = make_transforms(img_size)
    train_ds = TransformSubset(train_subset, train_tf)
    val_ds = TransformSubset(val_subset, eval_tf)
    sampler = None
    if use_rfs:
        sampler = WeightedRandomSampler(torch.as_tensor(repeat_factor_weights(TRAIN_TARGETS), dtype=torch.double),
                                        num_samples=len(train_ds), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=(sampler is None), sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True,
                              persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                            pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS > 0)
    return train_loader, val_loader


def make_test_loader(img_size):
    _, eval_tf = make_transforms(img_size)
    test_ds = COCOTestImageDataset(TEST_IMAGE_DIR, transform=eval_tf)
    return DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

### Split integrity check

The near-duplicate diagnostic above only looks for *similar* images. This cell checks the most basic property: that `train_ds[i]` really is the image `base_dataset[train_idx[i]]` (and the same for validation), with the same labels, and that the two index sets do not overlap. If one of the wrappers (`Subset`, `TransformSubset`) ignored the indices, validation images would silently be training images and every validation score of this report would be inflated. The check takes a few seconds.

In [ ]:
# 1) the two index sets must be disjoint
overlap = set(train_idx.tolist()) & set(val_idx.tolist())
print(f"Indices shared by train and validation : {len(overlap)}")

# 2) what the wrappers return must be what the base dataset returns for the same index (raw pixels, no transform)
try:
    identity = transforms.Lambda(lambda im: im)
    raw = {"train": (TransformSubset(train_subset, identity), train_idx),
           "validation": (TransformSubset(val_subset, identity), val_idx)}
    rng_check, mismatches = np.random.RandomState(0), 0
    for split_name, (wrapper, idx) in raw.items():
        for k in rng_check.choice(len(idx), size=min(20, len(idx)), replace=False):
            img_a, y_a = wrapper[int(k)]
            img_b, y_b = base_dataset[int(idx[k])]
            same_img = np.array_equal(np.asarray(img_a), np.asarray(img_b))
            same_lab = np.array_equal(np.asarray(y_a), np.asarray(y_b))
            if not (same_img and same_lab):
                mismatches += 1
                if mismatches <= 3:  # do not flood the output
                    print(f"  MISMATCH: {split_name} item {k} is not base_dataset[{idx[k]}]")
    verdict_ok = (len(overlap) == 0 and mismatches == 0)
    print("Split integrity check :", "OK" if verdict_ok else
    f"FAILED ({mismatches} mismatches) - the validation scores of this report are NOT trustworthy")
except Exception as exc:  # the check itself must never stop the notebook
    print("Split integrity check could not run :", repr(exc))

## 7. Metrics computation

**Primary metric: the leaderboard metric.** `inverse_frequency_weighted_f1` is kept exactly as defined in the starting notebook: the per-class precisions and recalls are averaged with weights proportional to `1 / n_positives`, then the F1 of these two averages is taken. It is the first number reported everywhere in this report (named `lb_f1_*`) because it is the closest to the leaderboard. Macro F1 and mAP are only secondary.

It is reported at four levels, from the simplest to the most elaborate:

| Name | Definition | How to use it |
|---|---|---|
| `lb_f1_05` | one threshold fixed at 0.5 for every class | no tuning at all: shows how good the raw probabilities are |
| `lb_f1_global` | best **single** threshold (grid 0.05-0.95) | headline number to compare models: one fitted parameter, so little over-fitting |
| `lb_f1_heldout` | per-class or blended thresholds fitted on half of validation and scored on the other half | honest estimate of the final pipeline |
| `lb_f1_per_class` | per-class thresholds fitted and scored on the same data | optimistic, for diagnosis only |

Tools:

- `pr_tables` pre-computes, for every class and every candidate threshold, the precision and the recall. The leaderboard metric is then just a weighted sum over classes, which makes it cheap to evaluate.
- `optimize_thresholds` starts from the best *global* threshold, then runs a coordinate ascent: classes are visited from the most to the least weighted, and a class threshold is changed only if the **global leaderboard F1 strictly improves**.
- Rare classes have very few positives in validation, so per-class thresholds can overfit. `cv_threshold_scores` measures this honestly: thresholds are fitted on one half of the validation set and scored on the other half. We compare three strategies (`global`, `per_class`, and `blend` = average of both) and keep the best one on held-out data.
- `apply_thresholds` can force at least one label per image (every COCO image has at least one) by taking the class with the highest `p / threshold`.
- `mAP` and an inverse-frequency-weighted `mAP` (`mAP_w`, threshold-free, same weighting as the leaderboard) are logged as secondary metrics.

In [ ]:
THR_GRID = np.round(np.arange(0.05, 0.951, 0.05), 3)


def compute_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(np.uint8)
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def inverse_frequency_weighted_f1(y_true, y_pred):
    positives = y_true.sum(axis=0)
    tp = ((y_true == 1) & (y_pred == 1)).sum(axis=0)
    fp = ((y_true == 0) & (y_pred == 1)).sum(axis=0)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    recall = np.divide(tp, positives, out=np.zeros_like(tp, dtype=float), where=positives > 0)
    weights = np.divide(1.0, positives, out=np.zeros_like(positives, dtype=float), where=positives > 0)
    if weights.sum() == 0: return 0.0
    weights = weights / weights.sum()
    p, r = np.sum(precision * weights), np.sum(recall * weights)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def class_weights(y_true):
    pos = y_true.sum(0).astype(float)
    w = np.divide(1.0, pos, out=np.zeros_like(pos), where=pos > 0)
    return w / w.sum()


def pr_tables(y_true, y_prob, grid=THR_GRID):
    """precision[c, t], recall[c, t] for every class c and candidate threshold t."""
    y = y_true.astype(bool)
    C, T = y.shape[1], len(grid)
    prec, rec = np.zeros((C, T)), np.zeros((C, T))
    pos = y.sum(0)
    for c in range(C):
        pred = y_prob[:, c, None] >= grid[None, :]
        tp = (pred & y[:, c, None]).sum(0)
        npred = pred.sum(0)
        prec[c] = np.divide(tp, npred, out=np.zeros(T), where=npred > 0)
        rec[c] = tp / pos[c] if pos[c] > 0 else 0.0
    return prec, rec


def _f1(P, R):
    return np.where(P + R > 0, 2 * P * R / np.maximum(P + R, 1e-12), 0.0)


def optimize_thresholds(y_true, y_prob, grid=THR_GRID, passes=3):
    """Coordinate ascent on the leaderboard metric. Returns (per-class thresholds, score, best global score)."""
    prec, rec = pr_tables(y_true, y_prob, grid)
    w = class_weights(y_true)
    C, T = prec.shape
    ar = np.arange(C)
    glob = np.array([_f1((w * prec[:, j]).sum(), (w * rec[:, j]).sum()) for j in range(T)])
    idx = np.full(C, int(np.argmax(glob)))
    best = float(glob.max())
    for _ in range(passes):
        P, R = (w * prec[ar, idx]).sum(), (w * rec[ar, idx]).sum()
        changed = False
        for c in np.argsort(-w):
            if w[c] == 0:
                continue
            Pn = P - w[c] * prec[c, idx[c]] + w[c] * prec[c]
            Rn = R - w[c] * rec[c, idx[c]] + w[c] * rec[c]
            F = _f1(Pn, Rn)
            j = int(np.argmax(F))
            if F[j] > best + 1e-9:
                idx[c], P, R, best, changed = j, Pn[j], Rn[j], float(F[j]), True
        if not changed:
            break
    return grid[idx], best, float(glob.max())


def optimize_global_threshold(y_true, y_prob, grid=THR_GRID):
    prec, rec = pr_tables(y_true, y_prob, grid)
    w = class_weights(y_true)
    scores = [_f1((w * prec[:, j]).sum(), (w * rec[:, j]).sum()) for j in range(len(grid))]
    return float(grid[int(np.argmax(scores))])


def fit_thresholds(y_true, y_prob, mode):
    C = y_true.shape[1]
    g = optimize_global_threshold(y_true, y_prob)
    if mode == "global":
        return np.full(C, g)
    pc, _, _ = optimize_thresholds(y_true, y_prob)
    return pc if mode == "per_class" else 0.5 * pc + 0.5 * g  # mode == "blend"


def apply_thresholds(y_prob, thr, force_one=True):
    pred = y_prob >= thr[None, :]
    if force_one:
        rows = np.flatnonzero(~pred.any(1))
        if len(rows):
            pred[rows, (y_prob[rows] / thr[None, :]).argmax(1)] = True
    return pred.astype(np.uint8)


def cv_threshold_scores(y_true, y_prob, n_repeats=3, seed=0):
    """Held-out leaderboard F1 of each (mode, force_one) strategy: fit on one half of the validation set, score on the other."""
    rng = np.random.RandomState(seed)
    scores = {}
    for _ in range(n_repeats):
        perm = rng.permutation(len(y_true))
        half = len(perm) // 2
        for a, b in ((perm[:half], perm[half:]), (perm[half:], perm[:half])):
            for mode in ("global", "blend", "per_class"):
                thr = fit_thresholds(y_true[a], y_prob[a], mode)
                for force in (False, True):
                    s = inverse_frequency_weighted_f1(y_true[b], apply_thresholds(y_prob[b], thr, force))
                    scores.setdefault((mode, force), []).append(s)
    return {k: float(np.mean(v)) for k, v in scores.items()}


def average_precisions(y_true, y_prob):
    aps = np.zeros(y_true.shape[1])
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() > 0:
            aps[c] = average_precision_score(y_true[:, c], y_prob[:, c])
    return aps


def lb_f1_at(y_true, y_prob, threshold=THRESHOLD):
    """Leaderboard metric (inverse-frequency-weighted F1) with one fixed threshold for every class."""
    return inverse_frequency_weighted_f1(y_true, (y_prob >= threshold).astype(np.uint8))


def epoch_metrics(y_true, y_prob):
    """Primary metrics first (leaderboard F1 at 0.5 / best global threshold / per-class thresholds), then mAP."""
    aps = average_precisions(y_true, y_prob)
    has_pos = y_true.sum(0) > 0
    _, lb_per_class, lb_global = optimize_thresholds(y_true, y_prob)
    return {"lb_f1_05": lb_f1_at(y_true, y_prob, THRESHOLD), "lb_f1_global": lb_global, "lb_f1_per_class": lb_per_class,
            "mAP": float(aps[has_pos].mean()), "mAP_w": float((aps * class_weights(y_true)).sum())}

## 8. Pretrained models and losses

All backbones are torchvision models whose last linear layer is replaced by an 80-output head. They can start from ImageNet weights or from random weights (see below). Two details matter for this imbalanced problem:

- **Head bias initialised to the log-odds of each class** (as in RetinaNet): at the start the model already predicts "rare" for rare classes instead of 0.5, which avoids a huge initial loss from the thousands of negatives.
- **Differential learning rates**: the new head uses `head_lr_mult` times the backbone learning rate.

Losses:
- `bce`: plain binary cross-entropy (baseline).
- `weighted_bce`: `pos_weight = neg / pos` (the original version). For a class with 100 positives that is a weight of about 650, which tends to wreck precision.
- `wbce_sqrt`: the same with a square root (weight about 25 for the rarest class), a much gentler version.
- `asl`: Asymmetric Loss (Ridnik et al., 2021), designed for multi-label problems where negatives vastly outnumber positives. It down-weights easy negatives (`gamma_neg = 4`, plus a probability margin `clip = 0.05`) while keeping the full gradient on positives (`gamma_pos = 0`). Note that ASL shifts the output probabilities, so the tuned thresholds below matter even more than with BCE.

**Three ways to start a ResNet18 (block A of section 10).**

- *Random init* (`pretrained=False`): the weights are random, the network has to learn everything from our 55 k images.
- *ImageNet, fine-tuned* (`pretrained=True`): the network starts from features learned on 1.2 M ImageNet images and **all layers** are trained, the new head with a 10x larger learning rate.
- *ImageNet, frozen* (`freeze_backbone=True`): same starting point, but only the new 80-output head is trained (about 41 k parameters instead of 11.2 M). The training loop also keeps the BatchNorm statistics of the backbone fixed (section 9), otherwise the backbone would keep changing although its weights are frozen.

### The methods used in this report and what they are expected to bring

This table states the *expectation* behind each method. Whether the expectation is confirmed by our runs is the job of section 16.

| Method | What it is | Expected advantage | Possible drawback |
|---|---|---|---|
| Random init | weights drawn at random | reference point: shows what our 55 k images alone can teach | needs far more data and epochs, weak on rare classes |
| ImageNet pre-training | start from weights learned on 1.2 M ImageNet images | generic visual features (edges, textures, object parts) are reused: faster convergence, better with few images per class | features learned for single-label ImageNet, not for cluttered COCO scenes |
| Fine-tuning | train all layers after pre-training | adapts the features to COCO, usually the best accuracy | more parameters to train (time, over-fitting), may forget useful features if the learning rate is too high |
| Frozen backbone (linear probe) | train only the new 80-output head | very cheap and fast, no over-fitting of the backbone, measures the quality of the raw ImageNet features | cannot adapt, lower ceiling |
| BCE | one sigmoid and one binary cross-entropy per class | natural multi-label loss | dominated by the many negatives: rare classes are under-predicted |
| Weighted BCE | positives weighted by `neg / pos` | pushes the recall of rare classes | extreme weights ruin the precision |
| ASL (Asymmetric Loss) | down-weights easy negatives, keeps full gradient on positives | designed for "negatives vastly outnumber positives" multi-label problems | shifts the probabilities, so thresholds must be tuned |
| Repeat-factor sampling | images with rare classes are repeated in an epoch | more gradient from rare classes, which carry most of the leaderboard weight | can over-fit the few rare images, precision may drop |
| OneCycle + EMA + flip TTA | training schedule, averaged weights, averaged predictions | smoother convergence and a small free gain | TTA doubles the inference cost |
| Threshold tuning on the leaderboard metric | thresholds chosen to maximise the inverse-frequency F1 | optimises what the leaderboard actually scores | per-class thresholds of rare classes rest on few positives, hence the split-half check |
| Ensemble | average of the probabilities of several runs | lower variance, small steady gain | several trainings and inferences |

In [ ]:
MODEL_REGISTRY = {
    # name: (constructor, weights enum, attribute holding the classification head)
    "resnet18": (models.resnet18, models.ResNet18_Weights, "fc"),
    "resnet50": (models.resnet50, models.ResNet50_Weights, "fc"),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights, "classifier"),
    "efficientnet_b3": (models.efficientnet_b3, models.EfficientNet_B3_Weights, "classifier"),
    "efficientnet_v2_s": (models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights, "classifier"),
    "convnext_tiny": (models.convnext_tiny, models.ConvNeXt_Tiny_Weights, "classifier"),
}


def replace_head(model, head_attr):
    head = getattr(model, head_attr)
    if isinstance(head, nn.Linear):
        new = nn.Linear(head.in_features, NUM_CLASSES)
        setattr(model, head_attr, new)
        return new
    last = len(head) - 1
    assert isinstance(head[last], nn.Linear), f"unexpected head for {head_attr}"
    head[last] = nn.Linear(head[last].in_features, NUM_CLASSES)
    return head[last]


def build_model(backbone="resnet18", pretrained=True, freeze_backbone=False):
    ctor, weights_enum, head_attr = MODEL_REGISTRY[backbone]
    model = ctor(weights=weights_enum.DEFAULT if pretrained else None)
    linear = replace_head(model, head_attr)
    with torch.no_grad():
        linear.bias.copy_(torch.as_tensor(PRIOR_LOGITS))
    if freeze_backbone:
        for name, p in model.named_parameters():
            if not name.startswith(head_attr + "."): p.requires_grad = False
    model = model.to(DEVICE)
    if CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model, head_attr


class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., ICCV 2021). Returns the loss summed over classes and averaged over the batch."""

    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, targets):
        logits = logits.float()
        p = torch.sigmoid(logits)
        p_neg = 1.0 - p
        if self.clip > 0:
            p_neg = (p_neg + self.clip).clamp(max=1.0)
        loss = targets * torch.log(p.clamp(min=self.eps)) + (1.0 - targets) * torch.log(p_neg.clamp(min=self.eps))
        with torch.no_grad():
            pt = p * targets + p_neg * (1.0 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            focus = torch.pow(1.0 - pt, gamma)
        return -(loss * focus).sum() / logits.shape[0]


def make_criterion(loss_name, train_targets=None):
    if loss_name == "bce":
        return nn.BCEWithLogitsLoss()
    if loss_name in ("weighted_bce", "wbce_sqrt"):
        pos = train_targets.sum(axis=0)
        ratio = (len(train_targets) - pos) / np.maximum(pos, 1)
        if loss_name == "wbce_sqrt":
            ratio = np.sqrt(ratio)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(ratio, dtype=torch.float32, device=DEVICE))
    if loss_name == "asl":
        return AsymmetricLoss(gamma_neg=4.0, gamma_pos=0.0, clip=0.05)
    raise ValueError(f"Unknown loss : {loss_name}")

## 9. Training and evaluation loop

Compared to the first version:

- **OneCycle schedule** (10% warm-up, cosine decay). With the old constant learning rate the validation loss was still falling at epoch 10, i.e. the model was stopped before converging, and the learning rate never decayed. There is no early stopping any more because it would cut the annealing; instead we keep the **best epoch**, selected on `SELECTION_METRIC` (default `lb_f1_global`: the leaderboard F1 with the best single threshold).
- **Mixed precision** (CUDA) and `channels_last` for speed, gradient clipping.
- **EMA** of the weights (usually a free gain for the final model); the EMA model is the one evaluated and saved.
- **Flip TTA** for the final validation/test predictions.
- Each run writes its validation/test probabilities to `OUTPUT_DIR` (`*_val_probs.npy`, `*_test_probs.npy`). Section 13 builds the final submission from these files, so experiments can be run one at a time (and resumed after a crash).
- **Frozen backbone** (`freeze_backbone=True`): the whole backbone is put in `eval()` mode (BatchNorm statistics, dropout and stochastic depth are fixed) and only the head is in `train()` mode. Without this, a "frozen" backbone would still drift through its BatchNorm statistics.
- Every epoch logs `lb_f1_05`, `lb_f1_global`, `lb_f1_per_class`, `mAP` and `mAP_w`; the summary of each run also stores whether it was pretrained / frozen and how many parameters were trained.

In [ ]:
class ModelEMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        src = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src[k].detach(), alpha=1 - d)
            else:
                v.copy_(src[k])


def _prep(images):
    images = images.to(DEVICE, non_blocking=True)
    return images.contiguous(memory_format=torch.channels_last) if CHANNELS_LAST else images


@torch.no_grad()
def predict_loader(model, loader, criterion=None, tta=False):
    model.eval()
    probs_all, targets_all, loss_sum, seen = [], [], 0.0, 0
    for images, targets in tqdm(loader, leave=False, desc="validation"):
        x = _prep(images)
        with amp_ctx():
            logits = model(x)
            logits_flip = model(torch.flip(x, dims=[3])) if tta else None
        logits = logits.float()
        probs = torch.sigmoid(logits)
        if tta:
            probs = 0.5 * (probs + torch.sigmoid(logits_flip.float()))
        if criterion is not None:
            loss_sum += criterion(logits, targets.to(DEVICE)).item() * x.size(0)
            seen += x.size(0)
        probs_all.append(probs.cpu().numpy())
        targets_all.append(targets.numpy())
    return np.concatenate(targets_all), np.concatenate(probs_all), loss_sum / max(seen, 1)


@torch.no_grad()
def predict_test(model, loader, tta=False):
    model.eval()
    ids, probs_all = [], []
    for images, names in tqdm(loader, leave=False, desc="test"):
        x = _prep(images)
        with amp_ctx():
            logits = model(x)
            logits_flip = model(torch.flip(x, dims=[3])) if tta else None
        probs = torch.sigmoid(logits.float())
        if tta:
            probs = 0.5 * (probs + torch.sigmoid(logits_flip.float()))
        probs_all.append(probs.cpu().numpy())
        ids.extend(list(names))
    return ids, np.concatenate(probs_all)


def set_train_mode(model, freeze_backbone, head_attr):
    """Training mode. A frozen backbone stays in eval mode (fixed BatchNorm statistics, no dropout); only the head trains."""
    if freeze_backbone:
        model.eval()
        getattr(model, head_attr).train()
    else:
        model.train()


def fit_experiment(name, backbone="resnet18", pretrained=True, freeze_backbone=False, loss_name="bce",
                   lr=1e-4, head_lr_mult=10.0, weight_decay=1e-2, img_size=224, use_rfs=False,
                   epochs=EPOCHS, use_ema=USE_EMA, label=None, group=None):
    set_seed()
    start = time.time()
    train_loader, val_loader = make_loaders(img_size, use_rfs)
    model, head_attr = build_model(backbone, pretrained, freeze_backbone)
    n_total = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = make_criterion(loss_name, TRAIN_TARGETS)

    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    groups_ = [{"params": [p for n, p in params if n.startswith(head_attr + ".")], "lr": lr * head_lr_mult}]
    body = [p for n, p in params if not n.startswith(head_attr + ".")]
    if body:
        groups_.append({"params": body, "lr": lr})
    optimizer = torch.optim.AdamW(groups_, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[g["lr"] for g in groups_], total_steps=epochs * len(train_loader),
        pct_start=0.1, anneal_strategy="cos", div_factor=20, final_div_factor=100, cycle_momentum=False)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    ema = ModelEMA(model) if use_ema else None

    best_score, best_state, best_epoch, history = -1.0, None, 0, []
    for epoch in range(epochs):
        set_train_mode(model, freeze_backbone, head_attr)
        running, seen = torch.zeros((), device=DEVICE), 0
        for images, targets in tqdm(train_loader, desc=f"{name} - epoch {epoch + 1}/{epochs}", leave=False):
            images, targets = _prep(images), targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_ctx():
                logits = model(images)
            loss = criterion(logits.float(), targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            if ema is not None:
                ema.update(model)
            running += loss.detach() * images.size(0)
            seen += images.size(0)

        eval_model = ema.module if ema is not None else model
        y_val, p_val, val_loss = predict_loader(eval_model, val_loader, criterion, tta=False)
        row = {"experiment": name, "epoch": epoch + 1, "train_loss": float(running) / max(seen, 1),
               "val_loss": val_loss, **epoch_metrics(y_val, p_val), "lr": optimizer.param_groups[-1]["lr"]}
        history.append(row)
        print({k: round(v, 4) if isinstance(v, (float, np.floating)) else v for k, v in row.items()})
        if row[SELECTION_METRIC] > best_score:
            best_score, best_epoch = row[SELECTION_METRIC], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in eval_model.state_dict().items()}
            torch.save({"state_dict": best_state, "backbone": backbone, "img_size": img_size, "loss_name": loss_name,
                        "classes": list(classes)}, OUTPUT_DIR / f"{name}_best.pt")

    # ---- final predictions of the best epoch (with TTA) ----
    eval_model = ema.module if ema is not None else model
    eval_model.load_state_dict(best_state)
    y_val, p_val, _ = predict_loader(eval_model, val_loader, None, tta=USE_TTA_FLIP)
    test_ids, p_test = predict_test(eval_model, make_test_loader(img_size), tta=USE_TTA_FLIP)

    np.save(OUTPUT_DIR / f"{name}_val_probs.npy", p_val.astype(np.float32))
    np.save(OUTPUT_DIR / f"{name}_test_probs.npy", p_test.astype(np.float32))
    targets_path, ids_path = OUTPUT_DIR / "val_targets.npy", OUTPUT_DIR / "test_ids.json"
    if targets_path.exists():
        assert np.array_equal(np.load(targets_path),
                              y_val), "validation split changed between runs - delete the cached files"
    else:
        np.save(targets_path, y_val.astype(np.uint8))
    if ids_path.exists():
        assert json.load(open(ids_path)) == list(test_ids), "test order changed between runs"
    else:
        json.dump(list(test_ids), open(ids_path, "w"))

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR / f"{name}_history.csv", index=False)
    summary = {"experiment": name, "label": label, "group": group, "backbone": backbone, "pretrained": pretrained,
               "freeze_backbone": freeze_backbone, "loss": loss_name, "img_size": img_size, "rfs": use_rfs,
               "lr": lr, "head_lr_mult": head_lr_mult, "epochs": epochs, "best_epoch": best_epoch,
               "selection_metric": SELECTION_METRIC, "params_total": n_total, "params_trained": n_trainable,
               **epoch_metrics(y_val, p_val), "seconds": time.time() - start}
    json.dump(summary, open(OUTPUT_DIR / f"{name}_summary.json", "w"), indent=2)

    del train_loader, val_loader, model, ema, eval_model, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return hist_df, summary

## 10. Experiments

Each experiment changes **one thing** compared with a reference run, so that a difference in score can be attributed to that thing. They are organised in three blocks.

**Block A - ResNet18: what do pre-training and freezing bring?** (BCE loss, 224 px, no resampling: only the starting point changes)

| Experiment | What changes | Question it answers |
|---|---|---|
| `r18_scratch_bce_224` | random weights, every layer trained | What does a ResNet18 reach with no prior knowledge, from 55 k images only? |
| `r18_bce_224` | ImageNet weights, every layer trained | How much does pre-training bring? (reference of block A) |
| `r18_frozen_bce_224` | ImageNet weights, only the head trained | Are the ImageNet features good enough as they are, and what is lost compared with fine-tuning? |

**Block B - Loss and sampling** (ResNet18 fine-tuned; reference: `r18_bce_224`)

| Experiment | What changes | Question it answers |
|---|---|---|
| `r18_asl_224` | ASL instead of BCE | Does a loss designed for rare positives help? |
| `r18_asl_rfs_224` | ASL + repeat-factor sampling | Does showing the rare classes more often help on top of ASL? |

**Block C - Backbones**

| Experiment | What changes | Question it answers |
|---|---|---|
| `r50_bce_224`, `effb0_bce_224` | other backbone, BCE, 224 px | Does a bigger or more efficient backbone beat ResNet18 at equal settings? (reference: `r18_bce_224`) |
| `effb3_asl_rfs_300`, `convnext_t_asl_rfs_288`, `effv2s_asl_rfs_300` | larger backbone and resolution with ASL + RFS | How far do the best loss/sampling settings go with a stronger model? (reference: `r18_asl_rfs_224`) |

Notes:

- The scratch run uses `lr=1e-3` and `head_lr_mult=1`: a randomly initialised network needs a larger learning rate than a fine-tuned one, and its head is not "newer" than the rest of the network. This gives the no-pre-training baseline a fair chance, but it means the scratch run differs by more than one factor from the reference. For a strict one-factor comparison, set `lr=2e-4` and `head_lr_mult=10`.
- `r18_bce_224`, `r18_asl_224` and `r18_asl_rfs_224` are the runs already trained for `rapport_v3` (same names, same settings): they are reused as they are if the `output/` folder is kept. The controlled ResNet study now also includes `r18_scratch_bce_224`, `r18_frozen_bce_224` and `r50_bce_224`.
- `RUN` selects the configurations trained in this session: finished experiments are skipped (their results are read from `OUTPUT_DIR`). Section 11 evaluates every finished run with complete probability files, even when it is not in `RUN`. Set `FORCE_RERUN = True` to retrain selected runs.

In [ ]:
EXPERIMENTS = [
    # ---------- Block A: ResNet18, BCE, 224 px - only the starting point changes ----------
    dict(name="r18_scratch_bce_224", label="ResNet18, random init", group="A", backbone="resnet18",
         pretrained=False, loss_name="bce", lr=1e-3, head_lr_mult=1.0, img_size=224),
    dict(name="r18_bce_224", label="ResNet18, ImageNet, fine-tuned", group="A", backbone="resnet18",
         pretrained=True, loss_name="bce", lr=2e-4, img_size=224),
    dict(name="r18_frozen_bce_224", label="ResNet18, ImageNet, frozen backbone", group="A", backbone="resnet18",
         pretrained=True, freeze_backbone=True, loss_name="bce", lr=2e-4, img_size=224),
    # ---------- Block B: loss and sampling (ResNet18 fine-tuned) ----------
    dict(name="r18_asl_224", label="ResNet18, ASL", group="B", backbone="resnet18",
         loss_name="asl", lr=2e-4, img_size=224),
    dict(name="r18_asl_rfs_224", label="ResNet18, ASL + repeat-factor sampling", group="B", backbone="resnet18",
         loss_name="asl", lr=2e-4, img_size=224, use_rfs=True),
    # ---------- Block C: other backbones ----------
    dict(name="r50_bce_224", label="ResNet50, BCE", group="C", backbone="resnet50",
         loss_name="bce", lr=2e-4, img_size=224),
    dict(name="effb0_bce_224", label="EfficientNet-B0, BCE", group="C", backbone="efficientnet_b0",
         loss_name="bce", lr=2e-4, img_size=224),
    dict(name="effb3_asl_rfs_300", label="EfficientNet-B3, ASL + RFS, 300 px", group="C", backbone="efficientnet_b3",
         loss_name="asl", lr=2e-4, img_size=300, use_rfs=True),
    dict(name="convnext_t_asl_rfs_288", label="ConvNeXt-Tiny, ASL + RFS, 288 px", group="C", backbone="convnext_tiny",
         loss_name="asl", lr=8e-5, weight_decay=5e-2, img_size=288, use_rfs=True),
    dict(name="effv2s_asl_rfs_300", label="EfficientNet-V2-S, ASL + RFS, 300 px", group="C",
         backbone="efficientnet_v2_s",
         loss_name="asl", lr=2e-4, img_size=300, use_rfs=True),
]
CONFIG = {cfg["name"]: cfg for cfg in EXPERIMENTS}
assert len(CONFIG) == len(EXPERIMENTS), "duplicate experiment names"

# The first study is deliberately limited to the controlled ResNet comparison.
# Section 11 still evaluates every finished run found in OUTPUT_DIR, including
# optional EfficientNet/ConvNeXt runs trained in a previous session.
RUN = [
    "r18_scratch_bce_224", "r18_bce_224", "r18_frozen_bce_224",
    "r18_asl_224", "r18_asl_rfs_224", "r50_bce_224",
]
FORCE_RERUN = False

for cfg in EXPERIMENTS:
    if cfg["name"] not in RUN:
        continue
    if (OUTPUT_DIR / f"{cfg['name']}_summary.json").exists() and not FORCE_RERUN:
        print(f"{cfg['name']} : already done, skipped")
        continue
    print(f"===== {cfg['name']} =====")
    fit_experiment(**cfg, epochs=EPOCHS)

## 11. Results visualization

This section is the evaluation gate for the report. **Every finished experiment with complete saved probabilities is evaluated and visualized**, including models trained in an earlier session or not selected for the final submission. The table is ranked by `lb_f1_global`, the leaderboard-style F1 with the best single threshold.

Every metric is recomputed from the saved validation probabilities (`*_val_probs.npy`), so all models are compared on exactly the same validation split and with exactly the same metric implementation. The two curves show the evolution of the primary metric and weighted mAP during training. No test prediction or submission is generated here.

`lb_f1_per_class` is optimistic (thresholds are tuned and scored on the same data). The honest comparison, with held-out thresholds, is made in section 12 and in the final comparative discussion.

In [ ]:
y_val = np.load(OUTPUT_DIR / "val_targets.npy")
all_summaries = {
    p.name[: -len("_summary.json")]: json.load(open(p))
    for p in sorted(OUTPUT_DIR.glob("*_summary.json"))
}

# Only runs with the complete validation and test artifacts can enter the
# comparison/ensemble. This makes the report safe to resume after an interrupted run.
summaries = {
    name: summary for name, summary in all_summaries.items()
    if (OUTPUT_DIR / f"{name}_val_probs.npy").exists()
       and (OUTPUT_DIR / f"{name}_test_probs.npy").exists()
}
skipped = sorted(set(all_summaries) - set(summaries))
if skipped:
    print("Runs skipped because probability files are incomplete:", skipped)
if not summaries:
    raise RuntimeError("No complete experiment found in OUTPUT_DIR. Run section 10 first.")


def run_metrics(name):
    """All metrics of a finished run, recomputed from its saved validation probabilities."""
    p = np.load(OUTPUT_DIR / f"{name}_val_probs.npy")
    _, lb_per_class, lb_global = optimize_thresholds(y_val, p)
    aps = average_precisions(y_val, p)
    return {"lb_f1_05": lb_f1_at(y_val, p, THRESHOLD), "lb_f1_global": lb_global, "lb_f1_per_class": lb_per_class,
            "mAP": float(aps[y_val.sum(0) > 0].mean()), "mAP_w": float((aps * class_weights(y_val)).sum())}


rows = []
for name, s_ in summaries.items():
    rows.append({"experiment": name, "model": CONFIG.get(name, {}).get("label", name), **run_metrics(name),
                 "best_epoch": s_.get("best_epoch"), "minutes": s_.get("seconds", float("nan")) / 60})
results_df = pd.DataFrame(rows).sort_values("lb_f1_global", ascending=False).reset_index(drop=True)
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for name in results_df["experiment"]:
    hist_path = OUTPUT_DIR / f"{name}_history.csv"
    if not hist_path.exists():
        continue
    hist = pd.read_csv(hist_path)
    lab = CONFIG.get(name, {}).get("label", name)
    axes[0].plot(hist["epoch"], hist["lb_f1_global"], marker="o", label=lab)
    axes[1].plot(hist["epoch"], hist["mAP_w"], marker="o", label=lab)
axes[0].set_title("Leaderboard F1, best global threshold (validation) - primary metric")
axes[1].set_title("Weighted mAP (validation) - secondary metric")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 12. Ensemble and threshold selection

The final prediction is the **average of the probabilities of the top-K runs** (ranked by the validation `lb_f1_global` from section 11), K = 1 ... 4. All runs are evaluated first; only after this comparison do we select the members used for the final prediction. For each K and each threshold strategy (`global`, `blend`, `per_class`, with or without forcing one label per image), we compute the **held-out** leaderboard F1 with the split-half procedure of section 7. We keep the (K, strategy) with the best held-out score, then refit the thresholds on the whole validation set.

This separation is intentional: the report can explain the performance of every model, while the submission is generated only from the objectively selected best configuration(s). The held-out score is the number to compare with the leaderboard (it is still computed on validation, so it is only as trustworthy as the split: see the duplicate diagnostic of section 6).

In [ ]:
y_val = np.load(OUTPUT_DIR / "val_targets.npy")
ranked = list(results_df["experiment"])
val_probs = {n: np.load(OUTPUT_DIR / f"{n}_val_probs.npy") for n in ranked}
test_probs = {n: np.load(OUTPUT_DIR / f"{n}_test_probs.npy") for n in ranked}

rows, best = [], None
for k in range(1, min(4, len(ranked)) + 1):
    members = ranked[:k]
    p = np.mean([val_probs[n] for n in members], axis=0)
    cv = cv_threshold_scores(y_val, p)
    for (mode, force), score in cv.items():
        rows.append({"K": k, "members": "+".join(members), "mode": mode, "force_one": force, "heldout_lb_f1": score})
        if best is None or score > best["score"]:
            best = {"score": score, "K": k, "members": members, "mode": mode, "force_one": force}
cv_df = pd.DataFrame(rows).sort_values("heldout_lb_f1", ascending=False)
display(cv_df.head(12))

FINAL_MEMBERS, FINAL_MODE, FORCE_ONE = best["members"], best["mode"], best["force_one"]
p_val_final = np.mean([val_probs[n] for n in FINAL_MEMBERS], axis=0)
final_thresholds = fit_thresholds(y_val, p_val_final, FINAL_MODE)
in_sample = inverse_frequency_weighted_f1(y_val, apply_thresholds(p_val_final, final_thresholds, FORCE_ONE))
print("Selected members :", FINAL_MEMBERS)
print("Threshold strategy :", FINAL_MODE, "| force one label :", FORCE_ONE)
print(f"Held-out leaderboard F1 (expected) : {best['score']:.4f} | in-sample : {in_sample:.4f}")
print("Thresholds (min / median / max) :", final_thresholds.min(), np.median(final_thresholds), final_thresholds.max())

## 13. Per-class metrics

Precision, recall and F1 of every class with the final thresholds, together with the weight of the class in the leaderboard metric. The classes with the highest `metric_weight` and a low F1 are where the next improvements are.

In [ ]:
y_pred = apply_thresholds(p_val_final, final_thresholds, FORCE_ONE)
w_val = class_weights(y_val)
per_class = []
for c, cname in enumerate(classes):
    per_class.append({
        "class": cname, "positifs_validation": int(y_val[:, c].sum()), "metric_weight": w_val[c],
        "precision": precision_score(y_val[:, c], y_pred[:, c], zero_division=0),
        "recall": recall_score(y_val[:, c], y_pred[:, c], zero_division=0),
        "f1": f1_score(y_val[:, c], y_pred[:, c], zero_division=0),
        "threshold": final_thresholds[c],
    })
per_class_df = pd.DataFrame(per_class)
per_class_df["f1_loss_x_weight"] = (1 - per_class_df["f1"]) * per_class_df["metric_weight"]
display(per_class_df.sort_values("f1_loss_x_weight", ascending=False).head(20))
per_class_df.to_csv(OUTPUT_DIR / "per_class_validation_metrics.csv", index=False)

## 14. Test submission

The test probabilities are generated only for the selected members from section 12. Their probabilities are averaged, the thresholds fitted during validation are applied, and the JSON file is written. All other models have already been evaluated and visualized in sections 11–13, but they do not contribute to the submission unless the selection procedure chooses them.

In [ ]:
test_ids = json.load(open(OUTPUT_DIR / "test_ids.json"))
p_test_final = np.mean([test_probs[n] for n in FINAL_MEMBERS], axis=0)
pred_test = apply_thresholds(p_test_final, final_thresholds, FORCE_ONE)

output_dict = {tid: np.flatnonzero(row).astype(int).tolist() for tid, row in zip(test_ids, pred_test)}
json_path = OUTPUT_DIR / "submission.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output_dict, f, ensure_ascii=False, indent=2)

print("JSON created :", json_path.resolve())
print("Number of predicted images :", len(output_dict))
print(f"Avg labels / image - true val : {y_val.sum(1).mean():.2f} | predicted val : {y_pred.sum(1).mean():.2f} "
      f"| predicted test : {pred_test.sum(1).mean():.2f}")
print("Example :", next(iter(output_dict.items())) if output_dict else "empty")

## 15. Submission verification

The submission file is checked independently of the training and comparison stages. It must contain exactly one entry for every **test** image, and every value must be a list of distinct valid class identifiers. This check uses the test directory (not the training directory).

In [ ]:
with open(json_path, "r", encoding="utf-8") as f:
    submission_check = json.load(f)

expected_ids = {p.stem for p in TEST_IMAGE_DIR.glob("*.jpg") if p.is_file()}
submitted_ids = set(submission_check)
assert submitted_ids == expected_ids, (
    f"images offset : {len(expected_ids - submitted_ids)} missing, "
    f"{len(submitted_ids - expected_ids)} unexpected. (If your ids are not file stems, adapt expected_ids.)"
)
for image_id, ids in submission_check.items():
    assert isinstance(image_id, str)
    assert isinstance(ids, list)
    assert all(isinstance(i, int) and 0 <= i < NUM_CLASSES for i in ids), image_id
    assert len(ids) == len(set(ids)), f"duplicate identifier {image_id}"
print("All tested passed.")
print("You can submit the file to the leaderboard :", json_path.resolve())

## 16. Comparative analysis of the models

This section compares the finished runs with each other and says **what changes when one thing changes**. It does not train anything: everything is recomputed from the files saved in `OUTPUT_DIR` (validation probabilities, training histories, summaries), so it can be re-run in a few minutes without any GPU, and it automatically takes into account the runs that exist at the time.

**How the comparison is made**

- Primary metric: the leaderboard F1 (inverse-frequency-weighted F1), at the best single threshold of each model (`lb_f1_global`) and at threshold 0.5 (`lb_f1_05`). Using a threshold fitted per model is fair: ASL, for instance, shifts the probabilities, so a common threshold of 0.5 would penalise it for a reason that has nothing to do with its ranking quality.
- Every block has a **reference run** and the table gives the difference to it (`Δ vs reference`) with a **95% bootstrap interval** (validation images resampled, same resample for the two models). A difference whose interval contains 0 is not distinguishable from noise.
- Classes are split in three groups of equal size by training frequency (**rare**, **medium**, **frequent**). Since the leaderboard metric weights a class by `1 / n_positives`, the rare group carries most of the score: the group F1 shows *where* a gain comes from.
- *Weighted precision* and *weighted recall* are the two numbers whose F1 is the leaderboard metric. They tell whether a method gains by finding more objects (recall) or by making fewer mistakes (precision).

**Limits to keep in mind.** One training per configuration (one seed): the intervals only account for the sampling noise of the validation set, not for the randomness of the training. The rarest classes have only 12 to 20 positive images in validation, and two of them (`hair drier`, `toaster`) carry about a quarter of the metric, so small differences are fragile. Section 16.5 explains how to check these validation numbers against the real leaderboard.

In [ ]:
import textwrap
from IPython.display import Markdown

y_val = np.load(OUTPUT_DIR / "val_targets.npy")
W_VAL = class_weights(y_val)
FINISHED = sorted(p.name[: -len("_summary.json")] for p in OUTPUT_DIR.glob("*_summary.json"))
SUMMARY = {n: json.load(open(OUTPUT_DIR / f"{n}_summary.json")) for n in FINISHED}
LABEL = {n: CONFIG.get(n, {}).get("label", n) for n in FINISHED}
VAL_P = {n: np.load(OUTPUT_DIR / f"{n}_val_probs.npy") for n in FINISHED}
print(f"{len(FINISHED)} finished runs :", ", ".join(FINISHED))

# classes split in three equal groups by number of positive images in the training folder
GROUP_NAMES = ["rare classes", "medium classes", "frequent classes"]
FREQ_GROUPS = np.array_split(np.argsort(class_counts), 3)
for g_name, g in zip(GROUP_NAMES, FREQ_GROUPS):
    print(f"  {g_name:<17s}: {len(g)} classes, {class_counts[g].min():,} to {class_counts[g].max():,} images each, "
          f"{W_VAL[g].sum():.0%} of the leaderboard metric")


def lb_pr(y_true, y_pred):
    """Weighted precision and recall used inside the leaderboard metric (the metric is the F1 of these two numbers)."""
    pos, npred = y_true.sum(0), y_pred.sum(0)
    tp = ((y_true == 1) & (y_pred == 1)).sum(0)
    prec = np.divide(tp, npred, out=np.zeros(len(tp)), where=npred > 0)
    rec = np.divide(tp, pos, out=np.zeros(len(tp)), where=pos > 0)
    w = class_weights(y_true)
    return float((prec * w).sum()), float((rec * w).sum())


def count_params(backbone, freeze_backbone):
    """(total, trained) number of parameters of a backbone with an 80-class head. No weight is downloaded."""
    try:
        ctor, _, head_attr = MODEL_REGISTRY[backbone]
        m = ctor(weights=None)
        replace_head(m, head_attr)
        total = sum(p.numel() for p in m.parameters())
        head = sum(p.numel() for p in getattr(m, head_attr).parameters())
        return total, (head if freeze_backbone else total)
    except Exception:
        return float("nan"), float("nan")


def evaluate_run(name):
    p = VAL_P[name]
    thr = optimize_global_threshold(y_val, p)
    pred = (p >= thr).astype(np.uint8)
    P, R = lb_pr(y_val, pred)
    f1_cls = f1_score(y_val, pred, average=None, zero_division=0)
    aps = average_precisions(y_val, p)
    cfg = CONFIG.get(name, {})
    total, trained = count_params(cfg.get("backbone", SUMMARY[name].get("backbone", "resnet18")),
                                  cfg.get("freeze_backbone", False))
    if total != total:  # NaN: torchvision unavailable, use what the training run saved (runs trained with v4 only)
        total, trained = SUMMARY[name].get("params_total", float("nan")), SUMMARY[name].get("params_trained",
                                                                                            float("nan"))
    return {"thr": thr, "lb_f1_05": lb_f1_at(y_val, p, THRESHOLD),
            "lb_f1_global": inverse_frequency_weighted_f1(y_val, pred),
            "P_w": P, "R_w": R, "mAP": float(aps[y_val.sum(0) > 0].mean()), "mAP_w": float((aps * W_VAL).sum()),
            "f1_macro_05": compute_metrics(y_val, p, THRESHOLD)["f1_macro"], "f1_cls": f1_cls,
            "group_f1": [float(f1_cls[g].mean()) for g in FREQ_GROUPS],
            "params_M": total / 1e6, "trained_M": trained / 1e6,
            "minutes": SUMMARY[name].get("seconds", float("nan")) / 60,
            "best_epoch": SUMMARY[name].get("best_epoch", float("nan"))}


EVAL = {n: evaluate_run(n) for n in FINISHED}

_BOOT = {}


def delta_ci(a, b, n_boot=200, seed=0):
    """lb_f1(a) - lb_f1(b) and its 95% bootstrap interval (validation images resampled, same resample for both models,
    each model at its own best global threshold)."""
    if (a, b) not in _BOOT:
        pa = (VAL_P[a] >= EVAL[a]["thr"]).astype(np.uint8)
        pb = (VAL_P[b] >= EVAL[b]["thr"]).astype(np.uint8)
        rng, n, diffs = np.random.RandomState(seed), len(y_val), []
        for _ in range(n_boot):
            idx = rng.randint(0, n, n)
            diffs.append(
                inverse_frequency_weighted_f1(y_val[idx], pa[idx]) - inverse_frequency_weighted_f1(y_val[idx], pb[idx]))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        _BOOT[(a, b)] = (EVAL[a]["lb_f1_global"] - EVAL[b]["lb_f1_global"], float(lo), float(hi))
    return _BOOT[(a, b)]


def verdict(a, b):
    d, lo, hi = delta_ci(a, b)
    if lo > 0:
        return f"better by {abs(d):.3f} (95% interval of the difference {lo:+.3f} to {hi:+.3f})"
    if hi < 0:
        return f"worse by {abs(d):.3f} (95% interval of the difference {lo:+.3f} to {hi:+.3f})"
    return f"not clearly different (difference {d:+.3f}, 95% interval {lo:+.3f} to {hi:+.3f})"


def compare_block(names, reference, title):
    """Table of differences to a reference run + headline metric, class-group F1 and learning curves."""
    names = [n for n in names if n in EVAL]
    if len(names) < 2 or reference not in names:
        print(
            f"{title}: not enough finished runs yet ({', '.join(names) or 'none'}). Train the runs of this block in section 10.")
        return
    rows = []
    for n in names:
        e = EVAL[n]
        row = {"model": LABEL[n], "lb_f1@0.5": e["lb_f1_05"], "lb_f1 (best global thr)": e["lb_f1_global"],
               "best thr": e["thr"]}
        if n == reference:
            row.update({"Δ vs reference": 0.0, "95% interval of Δ": "(reference)"})
        else:
            d, lo, hi = delta_ci(n, reference)
            row.update({"Δ vs reference": d, "95% interval of Δ": f"[{lo:+.3f}, {hi:+.3f}]"})
        row.update({"weighted precision": e["P_w"], "weighted recall": e["R_w"]})
        row.update({f"F1 {g}": v for g, v in zip(GROUP_NAMES, e["group_f1"])})
        row.update({"trained params (M)": e["trained_M"], "minutes": e["minutes"]})
        rows.append(row)
    display(pd.DataFrame(rows).round(4))

    x, wrapped = np.arange(len(names)), [textwrap.fill(LABEL[n], 20) for n in names]
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
    axes[0].bar(x - 0.2, [EVAL[n]["lb_f1_05"] for n in names], 0.4, label="threshold 0.5")
    axes[0].bar(x + 0.2, [EVAL[n]["lb_f1_global"] for n in names], 0.4, label="best global threshold")
    axes[0].set_title("Leaderboard F1 (validation)")
    for k, g in enumerate(GROUP_NAMES):
        axes[1].bar(x + (k - 1) * 0.27, [EVAL[n]["group_f1"][k] for n in names], 0.27, label=g)
    axes[1].set_title("F1 per group of classes (best global threshold)")
    for ax in axes[:2]:
        ax.set_xticks(x)
        ax.set_xticklabels(wrapped, fontsize=8)
        ax.legend(fontsize=8)
        ax.grid(True, axis="y", alpha=0.4)
    for n in names:
        hist_path = OUTPUT_DIR / f"{n}_history.csv"
        if hist_path.exists():
            hist = pd.read_csv(hist_path)
            axes[2].plot(hist["epoch"], hist["lb_f1_global"], marker="o", label=LABEL[n])
    axes[2].set_title("Leaderboard F1 during training (validation)")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(True)
    axes[2].legend(fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

    if len({CONFIG.get(n, {}).get("loss_name", "bce") for n in
            names}) == 1:  # losses are comparable only if the loss is the same
        fig, ax = plt.subplots(figsize=(8, 4))
        for n in names:
            hist_path = OUTPUT_DIR / f"{n}_history.csv"
            if hist_path.exists():
                hist = pd.read_csv(hist_path)
                line, = ax.plot(hist["epoch"], hist["val_loss"], marker="o", label=LABEL[n])
                ax.plot(hist["epoch"], hist["train_loss"], "--", color=line.get_color())
        ax.set_title("Loss: validation (solid) and training (dashed)")
        ax.set_xlabel("Epoch")
        ax.grid(True)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()


def top_class_differences(a, b, k=6):
    """Classes where model a gains / loses the most F1 compared with model b, ranked by (F1 difference x weight in the metric)."""
    d = EVAL[a]["f1_cls"] - EVAL[b]["f1_cls"]
    df = pd.DataFrame({"class": classes, "positives (val)": y_val.sum(0).astype(int), "metric weight": W_VAL,
                       "F1 (a)": EVAL[a]["f1_cls"], "F1 (b)": EVAL[b]["f1_cls"], "ΔF1": d, "ΔF1 x weight": d * W_VAL})
    print(f"(a) = {LABEL[a]}   |   (b) = {LABEL[b]}")
    gains, losses = df[df["ΔF1 x weight"] > 0], df[df["ΔF1 x weight"] < 0]
    print("Classes where (a) does better than (b), ranked by their effect on the leaderboard metric:")
    display(gains.sort_values("ΔF1 x weight", ascending=False).head(k).round(3) if len(gains) else "  none")
    print("Classes where (a) does worse than (b):")
    display(
        losses.sort_values("ΔF1 x weight").head(k).round(3) if len(losses) else "  none: (a) is not worse on any class")

### 16.1 What each run changes, and the overall results

The first table is the experimental design (one row per finished run). The second one is the leaderboard F1 of every model computed in four ways (see section 7); the last columns give the weighted precision and recall at the best global threshold, and the secondary metrics. The chart shows how much of the score depends on the choice of thresholds: a model whose `lb_f1@0.5` is far below its `lb_f1 (best global thr)` has probabilities that are not calibrated for a 0.5 cut (typical of ASL), which says nothing about its ability to rank images correctly.

In [ ]:
design = pd.DataFrame([{
    "block": CONFIG[n].get("group", "?"), "experiment": n, "model": LABEL[n], "backbone": CONFIG[n]["backbone"],
    "init": "ImageNet" if CONFIG[n].get("pretrained", True) else "random",
    "trained layers": "head only" if CONFIG[n].get("freeze_backbone", False) else "all",
    "loss": CONFIG[n].get("loss_name", "bce"),
    "sampling": "repeat-factor" if CONFIG[n].get("use_rfs", False) else "uniform",
    "img size": CONFIG[n].get("img_size", 224), "lr": f"{CONFIG[n].get('lr', float('nan')):g}",
    "head lr x": CONFIG[n].get("head_lr_mult", 10.0),
    "params (M)": EVAL[n]["params_M"], "trained (M)": EVAL[n]["trained_M"]} for n in FINISHED if n in CONFIG])
display(design.sort_values(["block", "experiment"]).round(2))

print(
    "Held-out scores (thresholds fitted on one half of validation, scored on the other half) - this takes a minute...")
HELDOUT = {n: cv_threshold_scores(y_val, VAL_P[n]) for n in FINISHED}

rows = []
for n in FINISHED:
    e, h = EVAL[n], HELDOUT[n]
    rows.append(
        {"experiment": n, "model": LABEL[n], "lb_f1@0.5": e["lb_f1_05"], "lb_f1 (best global thr)": e["lb_f1_global"],
         "best thr": e["thr"], "lb_f1 held-out, global thr": h[("global", False)],
         "lb_f1 held-out, per-class thr": h[("per_class", False)], "weighted precision": e["P_w"],
         "weighted recall": e["R_w"], "mAP": e["mAP"], "mAP_w": e["mAP_w"], "macro F1@0.5": e["f1_macro_05"],
         "best epoch": e["best_epoch"], "minutes": e["minutes"]})
metrics_df = pd.DataFrame(rows).sort_values("lb_f1 (best global thr)", ascending=False).reset_index(drop=True)
display(metrics_df.round(4))
metrics_df.to_csv(OUTPUT_DIR / "comparison_all_models.csv", index=False)

cols = ["lb_f1@0.5", "lb_f1 (best global thr)", "lb_f1 held-out, global thr", "lb_f1 held-out, per-class thr"]
fig, ax = plt.subplots(figsize=(11, 0.6 * len(metrics_df) + 2))
yy, hh = np.arange(len(metrics_df)), 0.2
for k, c in enumerate(cols):
    ax.barh(yy + (k - 1.5) * hh, metrics_df[c], height=hh, label=c)
ax.set_yticks(yy)
ax.set_yticklabels(metrics_df["model"])
ax.invert_yaxis()
ax.set_xlabel("Leaderboard F1 (validation)")
ax.grid(True, axis="x", alpha=0.4)
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

### 16.2 Block A - ResNet18: random weights, ImageNet weights, frozen backbone

**What to look at.** Three comparisons share one reference (`ResNet18, ImageNet, fine-tuned`):

- *Random init vs ImageNet.* This isolates the value of **transfer learning**. The expectation is a clearly lower score for random weights, mostly on the **rare** classes (a few hundred images cannot teach a network from zero what a toaster looks like), and a learning curve that starts lower and climbs more slowly.
- *Frozen vs fine-tuned.* This isolates the value of **adapting the backbone**. The frozen run trains about 41 k parameters instead of 11.2 M and should be much cheaper; the question is how much score it gives up. A frozen backbone cannot adapt its features to COCO, so a lower ceiling is expected, mostly on the classes that ImageNet does not cover well.
- *Random init vs frozen* shows which is worth more here: ImageNet knowledge that cannot adapt, or full freedom with no knowledge.

**Sanity check.** A ResNet18 trained from random weights on 55 k images should be far from the pre-trained one. If the random-init run lands close to the pre-trained run, the validation set is probably not independent from the training set (see the integrity check of section 6 and section 16.5).

In [ ]:
BLOCK_A = ["r18_scratch_bce_224", "r18_bce_224", "r18_frozen_bce_224"]
compare_block(BLOCK_A, reference="r18_bce_224",
              title="Block A - ResNet18: random init vs ImageNet fine-tuned vs ImageNet frozen")

if all(n in EVAL for n in ("r18_bce_224", "r18_scratch_bce_224")):
    print("\n=== Pre-training: which classes gain? ===")
    top_class_differences("r18_bce_224", "r18_scratch_bce_224")
if all(n in EVAL for n in ("r18_bce_224", "r18_frozen_bce_224")):
    print("\n=== Fine-tuning: which classes gain compared with a frozen backbone? ===")
    top_class_differences("r18_bce_224", "r18_frozen_bce_224")

### 16.3 Block B - Loss and sampling (ResNet18, fine-tuned)

**What to look at.** The reference is plain BCE.

- *ASL vs BCE.* ASL down-weights the many easy negatives. Expect a **different best threshold** (ASL pushes probabilities up, so the best threshold moves away from 0.5 and `lb_f1@0.5` falls below `lb_f1 (best global thr)`) and possibly a higher weighted **recall**. The fair comparison is the one at the best threshold of each model.
- *Repeat-factor sampling (RFS) on top of ASL.* RFS shows the rare classes more often. Expect a gain in the **rare** group and in recall, possibly paid in precision or in the frequent group. The class tables below show which classes move.

In [ ]:
BLOCK_B = ["r18_bce_224", "r18_asl_224", "r18_asl_rfs_224"]
compare_block(BLOCK_B, reference="r18_bce_224", title="Block B - ResNet18: BCE vs ASL vs ASL + repeat-factor sampling")

if all(n in EVAL for n in ("r18_asl_rfs_224", "r18_asl_224")):
    print("\n=== Repeat-factor sampling (on top of ASL): which classes move? ===")
    top_class_differences("r18_asl_rfs_224", "r18_asl_224")

### 16.4 Block C - Other backbones (only if these runs were trained)

Two separate questions, each with its own reference: other backbones at the settings of the BCE baseline (reference `ResNet18, ImageNet, fine-tuned`), and larger backbones at higher resolution with the best loss/sampling (reference `ResNet18, ASL + repeat-factor sampling`). Read the gain together with the cost: trained parameters and minutes are in the tables.

In [ ]:
compare_block(["r18_bce_224", "r50_bce_224", "effb0_bce_224"], reference="r18_bce_224",
              title="Block C1 - other backbones at the BCE baseline settings")
compare_block(["r18_asl_rfs_224", "effb3_asl_rfs_300", "convnext_t_asl_rfs_288", "effv2s_asl_rfs_300"],
              reference="r18_asl_rfs_224", title="Block C2 - larger backbones and resolution with ASL + RFS")

### 16.5 Can the validation numbers be trusted? Validation vs test vs leaderboard

Every comparison above is made on the validation set, so it is only as good as the validation set. Three checks:

1. **Plausibility.** The cell warns if the best validation mAP is far above what is published for this task.
2. **Behaviour on test.** For each model, the average number of labels it predicts per image on validation and on test, with the same threshold. The true average is about 2.9 labels per image. A model that predicts clearly fewer labels on test than on validation is facing images that are harder than its validation set.
3. **The leaderboard itself.** It is the only ground truth. Set `WRITE_PER_MODEL_SUBMISSIONS = True` to write one submission file per model (its own best global threshold, at least one label per image), submit them, and paste the scores in `LEADERBOARD_SCORES`. The cell then shows the gap validation - leaderboard and whether the *ranking* of the models is the same on both. A validation set that gets the ranking right is still useful for choosing between methods even if its absolute level is too high.

In [ ]:
LEADERBOARD_SCORES = {
    # paste here the score given by the leaderboard to the submission made with this model, for example:
    # "r18_bce_224": 0.44,
}
WRITE_PER_MODEL_SUBMISSIONS = False


def write_submission_for(name):
    ids = json.load(open(OUTPUT_DIR / "test_ids.json"))
    p_test = np.load(OUTPUT_DIR / f"{name}_test_probs.npy")
    pred = apply_thresholds(p_test, np.full(NUM_CLASSES, EVAL[name]["thr"]), force_one=True)
    out = {tid: np.flatnonzero(row).astype(int).tolist() for tid, row in zip(ids, pred)}
    path = OUTPUT_DIR / f"submission_{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f)
    return path


if WRITE_PER_MODEL_SUBMISSIONS:
    for n in FINISHED:
        print("written :", write_submission_for(n))

rows = []
for n in FINISHED:
    e = EVAL[n]
    test_path = OUTPUT_DIR / f"{n}_test_probs.npy"
    row = {"model": LABEL[n], "val lb_f1 (best global thr)": e["lb_f1_global"], "val mAP": e["mAP"],
           "best thr": e["thr"],
           "labels/img val (true)": y_val.sum(1).mean(),
           "labels/img val (predicted)": (VAL_P[n] >= e["thr"]).sum(1).mean(),
           "labels/img test (predicted)": (np.load(test_path) >= e["thr"]).sum(
               1).mean() if test_path.exists() else float("nan")}
    if n in LEADERBOARD_SCORES:
        row.update(
            {"leaderboard": LEADERBOARD_SCORES[n], "val - leaderboard": e["lb_f1_global"] - LEADERBOARD_SCORES[n]})
    rows.append(row)
gap_df = pd.DataFrame(rows).sort_values("val lb_f1 (best global thr)", ascending=False).reset_index(drop=True)
display(gap_df.round(3))

if len(LEADERBOARD_SCORES) >= 3:
    sub = gap_df.dropna(subset=["leaderboard"])
    rank_corr = sub["val lb_f1 (best global thr)"].rank().corr(sub["leaderboard"].rank())
    print(f"Rank correlation between validation and leaderboard over {len(sub)} models : {rank_corr:+.2f} "
          "(+1 = same ranking, 0 = unrelated)")
elif LEADERBOARD_SCORES:
    print("Add the leaderboard score of at least 3 models to get a rank correlation.")
else:
    print("No leaderboard score entered yet (LEADERBOARD_SCORES is empty).")

best_map = max(e["mAP"] for e in EVAL.values())
if best_map > 0.80:
    print(
        f"\nWARNING: the best validation mAP is {best_map:.3f}. Published results on COCO multi-label classification are "
        "roughly 0.78-0.90 mAP, obtained with much larger backbones at 448 px or more; a ResNet18 at 224 px should be well "
        "below. Such a high value usually means that validation images were seen during training. Run the split integrity "
        "check of section 6 and compare with the leaderboard before drawing conclusions from the validation numbers.")

### 16.6 Findings

The text below is **generated from the numbers above** each time the notebook is run, so it always matches the current results. "Better" and "worse" are only used when the 95% interval of the difference excludes 0; otherwise the runs are reported as not clearly different.

In [ ]:
SCR, PRE, FRZ = "r18_scratch_bce_224", "r18_bce_224", "r18_frozen_bce_224"
ASL, RFS = "r18_asl_224", "r18_asl_rfs_224"


def have(*names):
    return all(n in EVAL for n in names)


out = []
best = max(EVAL, key=lambda n: EVAL[n]["lb_f1_global"])
out.append(
    f"- **Best single model on validation:** {LABEL[best]} (`{best}`), leaderboard-style F1 {EVAL[best]['lb_f1_global']:.3f} "
    f"at threshold {EVAL[best]['thr']:.2f} ({EVAL[best]['lb_f1_05']:.3f} at 0.5), weighted precision "
    f"{EVAL[best]['P_w']:.3f} and weighted recall {EVAL[best]['R_w']:.3f}.")

if have(PRE, SCR):
    a, b = EVAL[PRE], EVAL[SCR]
    out.append(
        f"- **Pre-training.** ImageNet weights vs random weights (same architecture, same loss): ImageNet is {verdict(PRE, SCR)}. "
        f"Weighted recall {b['R_w']:.3f} -> {a['R_w']:.3f}, F1 on the rare classes {b['group_f1'][0]:.3f} -> {a['group_f1'][0]:.3f}, "
        f"on the frequent classes {b['group_f1'][2]:.3f} -> {a['group_f1'][2]:.3f}.")
if have(FRZ, PRE):
    a, b = EVAL[FRZ], EVAL[PRE]
    out.append(
        f"- **Freezing the backbone.** Frozen vs fully fine-tuned: the frozen run is {verdict(FRZ, PRE)}. The frozen run trains {a['trained_M']:.2f} M "
        f"parameters instead of {b['trained_M']:.1f} M and took {a['minutes']:.0f} instead of {b['minutes']:.0f} minutes. "
        f"F1 on the rare classes {b['group_f1'][0]:.3f} (fine-tuned) vs {a['group_f1'][0]:.3f} (frozen).")
if have(FRZ, SCR):
    out.append(f"- **Frozen ImageNet features vs no pre-training at all:** frozen is {verdict(FRZ, SCR)}.")
if have(ASL, PRE):
    a, b = EVAL[ASL], EVAL[PRE]
    out.append(
        f"- **ASL vs BCE.** ASL is {verdict(ASL, PRE)}. Best threshold {b['thr']:.2f} (BCE) -> {a['thr']:.2f} (ASL); at the fixed threshold 0.5 "
        f"the leaderboard F1 goes from {b['lb_f1_05']:.3f} to {a['lb_f1_05']:.3f}. Weighted recall {b['R_w']:.3f} -> {a['R_w']:.3f}, "
        f"weighted precision {b['P_w']:.3f} -> {a['P_w']:.3f}.")
if have(RFS, ASL):
    a, b = EVAL[RFS], EVAL[ASL]
    out.append(
        f"- **Repeat-factor sampling on top of ASL.** Adding it is {verdict(RFS, ASL)}. F1 on the rare classes {b['group_f1'][0]:.3f} -> {a['group_f1'][0]:.3f}, "
        f"on the frequent classes {b['group_f1'][2]:.3f} -> {a['group_f1'][2]:.3f}; weighted recall {b['R_w']:.3f} -> {a['R_w']:.3f}, "
        f"weighted precision {b['P_w']:.3f} -> {a['P_w']:.3f}.")
for n in ("r50_bce_224", "effb0_bce_224"):
    if have(n, PRE):
        out.append(
            f"- **{LABEL[n]} vs ResNet18 (BCE, 224 px).** It is {verdict(n, PRE)}; {EVAL[n]['params_M']:.1f} M parameters instead of "
            f"{EVAL[PRE]['params_M']:.1f} M, {EVAL[n]['minutes']:.0f} minutes instead of {EVAL[PRE]['minutes']:.0f}.")
for n in ("effb3_asl_rfs_300", "convnext_t_asl_rfs_288", "effv2s_asl_rfs_300"):
    if have(n, RFS):
        out.append(
            f"- **{LABEL[n]} vs ResNet18 ASL + RFS.** It is {verdict(n, RFS)}; {EVAL[n]['params_M']:.1f} M parameters instead of "
            f"{EVAL[RFS]['params_M']:.1f} M, {EVAL[n]['minutes']:.0f} minutes instead of {EVAL[RFS]['minutes']:.0f}.")

if max(e["mAP"] for e in EVAL.values()) > 0.80:
    out.append(
        "- **Warning on the validation set.** The validation mAP is higher than what is published for much larger models "
        "(section 16.5): the *relative* comparisons above are only reliable if the split integrity check of section 6 passed "
        "and, ideally, if the ranking of the models is confirmed by the leaderboard.")
out.append(
    "- **Reminder.** One training per configuration: the intervals only cover the sampling noise of the validation set, "
    "not the randomness of the training. Re-run a configuration with another `SEED` before relying on a small difference.")
display(Markdown("\n".join(out)))

### 16.7 How to turn these findings into the conclusion of the report

For each block, the conclusion has the same three parts: **what changed** (one factor, see section 10), **what we observed** (the sentences of 16.6 and the tables above, with the interval), and **why it is plausible** (the expected advantage of the method, section 8, confirmed or contradicted by the numbers). When an expectation is contradicted, say so: it is a result as well. Finally, state which configuration was kept for the submission (section 12) and on which evidence, and mention the limits listed at the beginning of this section.

## 17. Discussion: choix des modèles et des approches

Cette partie met en relation les résultats obtenus avec les choix méthodologiques. Les modèles ne sont pas comparés uniquement selon leur score final : le coût d'entraînement, le nombre de paramètres, la stabilité des courbes et le comportement sur les classes rares sont également pris en compte.

| Approche | Intérêt principal | Limite attendue | Rôle dans l'étude |
|---|---|---|---|
| ResNet18 initialisé aléatoirement | Référence sans transfert d'apprentissage | Apprentissage plus lent et représentation moins adaptée au début | Mesurer l'apport réel des poids ImageNet |
| ResNet18 pré-entraîné puis entièrement affiné | Bon compromis entre coût et adaptation au domaine COCO | Capacité limitée par rapport à un backbone plus profond | Modèle de référence principal |
| ResNet18 avec backbone gelé | Entraînement rapide et peu de paramètres optimisés | Les caractéristiques ImageNet ne peuvent pas s'adapter aux classes COCO | Mesurer le bénéfice du fine-tuning |
| ResNet50 pré-entraîné puis affiné | Réseau plus profond, représentation potentiellement plus riche | Temps et mémoire supérieurs, risque de gain marginal | Vérifier si la capacité supplémentaire est utile |
| ASL et repeat-factor sampling | Accorder davantage d'attention aux positifs rares et aux exemples difficiles | Probabilités moins calibrées et risque de sur-apprentissage des classes très rares | Tester une réponse au déséquilibre des classes |
| EfficientNet / ConvNeXt | Meilleur compromis possible entre résolution, capacité et efficacité | Comparaison moins isolée lorsque la résolution et la loss changent aussi | Explorer des architectures alternatives après le contrôle ResNet |

Le protocole commence par ResNet18 car il permet d'isoler proprement trois facteurs : l'initialisation, le gel du backbone et le fine-tuning. ResNet50 est ensuite comparé à la référence avec la même résolution, la même loss et le même protocole d'évaluation. Cette organisation évite d'attribuer à l'architecture un gain qui proviendrait en réalité d'une autre résolution, d'une autre loss ou d'un autre échantillonnage.

Le choix final est effectué après l'évaluation de tous les modèles disponibles sur la validation. Les tableaux et courbes des sections 11 à 13 restent donc nécessaires même si une seule configuration, ou un petit ensemble des meilleures configurations, est utilisé pour la submission. La submission ne constitue pas une étape d'évaluation des modèles : elle matérialise uniquement la sélection faite à partir des résultats précédents.

Pour la rédaction de la conclusion, il faut commenter séparément : (1) l'apport du pré-entraînement en comparant ResNet18 aléatoire et ImageNet, (2) l'apport du fine-tuning en comparant le backbone gelé et le backbone entièrement entraîné, (3) le rapport coût/performance de ResNet50 face à ResNet18, puis (4) l'intérêt réel d'ASL et du repeat-factor sampling sur les classes rares. Une amélioration dont l'intervalle de confiance contient zéro doit être présentée comme non concluante plutôt que comme une preuve de supériorité.